In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import wandb
from datasets import get_dataset
import torch
import random
import numpy as np
from utils import load_run, show_img_chw, get_image_chw
from funcs import train_epoch, val_epoch, test
from transforms import get_transforms

from models.switch.utils import set_activate_n_expert, set_save_token_routes
from constants import get_classes
from vit_configs import get_vit_config
from loaders import get_loaders
from transformers import ViTForImageClassification
from pytorchvideo.transforms import MixUp
from models.models_functions import get_model, init_model, model_to_switch
import matplotlib.pyplot as plt

In [ ]:
RUN_ID = "qwectpiy"
EPOCH_NR = 1
ACTIVATE_ONLY_N_EXPERT = None

In [ ]:
SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

wandb.login(key="25a61b596081a070a75d3cfc16bfad96632650fc")
wandb.init(project="msc", resume=True, id=RUN_ID)
config = wandb.config

config['batch_size'] = 1

if "switch_top_k" not in config:
    config["switch_top_k"] = 1
run_model_name = wandb.run.name

CLASSES_LIST = get_classes(config["dataset"])

VIT_CONFIG = get_vit_config(config["vit_base_size"])

ds_transform_train, ds_transform_valid, ds_transform_test = get_transforms(config, CLASSES_LIST, config["dataset"])

dataset_train, dataset_valid, dataset_test, num_labels = get_dataset(
    config["dataset"],
    ds_transform_train,
    ds_transform_valid,
    ds_transform_test,
    seconds=config["clip_duration"],
    subsample=config["subsample"],
)

vit = ViTForImageClassification.from_pretrained(f"WinKawaks/vit-{config['vit_base_size']}-patch16-224")

model, model_config = get_model(config["model"], num_labels, VIT_CONFIG, config)

if config["switch"]:
    model = model_to_switch(config["model"], model, config, model_config)

model.to(device)

if config["init_vit"]:
    model = init_model(config["model"], model, vit, VIT_CONFIG["num_hidden_layers"], config)

training_loader, validation_loader, test_loader = get_loaders(
    dataset_train, dataset_valid, dataset_test, SEED, config["batch_size"], n_workers=1
)

epochs = config["epochs"]
label_smoothing = config["label_smoothing"]

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=config["base_learning_rate"],
    momentum=0.9,
    weight_decay=config["weight_decay"],
)

loss = torch.nn.functional.cross_entropy


model, optimizer, prev_epoch = load_run(model, optimizer, run_model_name, EPOCH_NR)

if ACTIVATE_ONLY_N_EXPERT and config["switch"]:
    model = set_activate_n_expert(model, ACTIVATE_ONLY_N_EXPERT, config["switch_layers"])

model = set_save_token_routes(model, config, save_token_routes=True)

In [ ]:
with torch.no_grad():
    for data in validation_loader:
        images = data["video"]
        labels = data["label"]
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images).logits
        break

In [ ]:
color_map = {
    0: [0.8, 1.4, 0.8],  # Greener
    1: [1.4, 0.8, 0.8],  # Redder
    2: [0.8, 0.8, 1.4],  # Bluer
    3: [1.4, 1.4, 0.8],  # Yellower
    4: [0.8, 1.4, 1.4],  # Cyaner
    5: [1.4, 0.8, 1.4]   # Magenter
}
PATCH_WIDTH= 16
PATCH_HEIGHT = 16
def apply_filter(img, row_nr, col_nr, expert_nr):
    image_filtered = img.copy()

    y_start = row_nr * PATCH_HEIGHT
    x_start = col_nr * PATCH_WIDTH
    y_end = y_start + PATCH_HEIGHT
    x_end = x_start + PATCH_WIDTH

    patch = image_filtered[y_start:y_end, x_start:x_end]

    factors = np.array(color_map[expert_nr], dtype=np.float32)

    filtered_patch_data = patch.astype(np.float32) * factors
    filtered_patch_data = np.clip(filtered_patch_data, 0, 1)
    filtered_patch = filtered_patch_data.astype(np.float32)     # Convert back to uint8

    image_filtered[y_start:y_end, x_start:x_end] = filtered_patch
    return image_filtered

In [ ]:
PATCHES_IN_ROW = 14
N_FRAMES = 8


frame_nr = 0
img = get_image_chw(images[0][0].cpu(), denormalize=True).numpy()
plt.figure()
plt.imshow(img)
plt.title("Original")


for layer_nr in config['switch_layers']:
    img = get_image_chw(images[0][0].cpu(), denormalize=True).numpy()
    routes = model.timesformer.encoder.layer[layer_nr].intermediate.token_routes.cpu().numpy()
    routes = routes[1:]     # first is cls


    for row in range(PATCHES_IN_ROW):  # n_patches
        for col in range(PATCHES_IN_ROW): # n_patches
            spatial_patch_idx = row * PATCHES_IN_ROW + col
            start_index_for_spatial_patch_block = spatial_patch_idx * N_FRAMES
            token_index = start_index_for_spatial_patch_block + frame_nr
            route = routes[token_index]
            img = apply_filter(img, row, col, route)
    plt.figure()
    plt.imshow(img)
    plt.title(f"Layer {layer_nr + 1}")

In [ ]:
routes = routes[1:]